# Q9.
```{admonition}
:class: note
Fit a lag-5 autoregressive model to the `NYSE` data. Refit the model with a 12-level factor representing the month. Does this factor improve the performance of the model?

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
nyse = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/NYSE.csv', parse_dates=['date'], index_col=['date'], usecols=['date','DJ_return','log_volume','log_volatility']).sort_index()
nyse_train, nyse_test = train_test_split(nyse, shuffle=False, train_size=4276)

In [4]:
scaler = StandardScaler()
nyse_train[nyse.columns] = scaler.fit_transform(nyse_train)
nyse_test[nyse.columns] = scaler.transform(nyse_test)
nyse_scaled = pd.concat([nyse_train,nyse_test])

weekday_dummy = pd.get_dummies(nyse_scaled.index.day_name()).set_index(nyse_scaled.index)

nyse_week_lag = pd.concat([nyse_scaled]+[nyse_scaled.shift(lag).add_suffix(f'_lag{lag}') for lag in range (1,6)], axis=1).dropna()
nyse_week_lag = nyse_week_lag.join(weekday_dummy)

In [5]:
Y = nyse_week_lag['log_volume']
X = nyse_week_lag.drop(columns=nyse_scaled.columns)

X_train, X_test, y_train, y_test = train_test_split(X, Y, shuffle=False, train_size=4276)

In [6]:
lr = LinearRegression()
lr.fit(X_train,y_train)
week_mse = mean_squared_error(y_test,lr.predict(X_test))
week_r2 = r2_score(y_test,lr.predict(X_test))
print(f'Test MSE: {week_mse:.4f}')
print(f'Test R2 score: {week_r2:.4f}')

Test MSE: 0.5837
Test R2 score: 0.4596


In [8]:
month_dummy = pd.get_dummies(nyse_scaled.index.month_name()).set_index(nyse_scaled.index)

nyse_month_week_lag = nyse_week_lag.join(month_dummy)

In [9]:
Y = nyse_month_week_lag['log_volume']
X = nyse_month_week_lag.drop(columns=nyse_scaled.columns)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, shuffle=False, train_size=4276)

In [11]:
lrm = LinearRegression()
lrm.fit(X_train,y_train)
month_mse = mean_squared_error(y_test,lrm.predict(X_test))
month_r2 = r2_score(y_test,lrm.predict(X_test))
print(f'Test MSE: {month_mse:.4f}')
print(f'Test R2 score: {month_r2:.4f}')

Test MSE: 0.5803
Test R2 score: 0.4627
